# Extraction EDA

Run `uv run -m uni_rag_agent extract run` after inventory has populated `data/uni_rag.sqlite`, then use this notebook to inspect extraction yield, failures, chunk counts, source types, and source-location coverage.

Safety boundary: this notebook is read-only. It reads generated app data only, opens SQLite in read-only mode, enables `PRAGMA query_only=ON`, and must not mutate SQLite, `Courses`, or any course file.

In [4]:
from pathlib import Path
import sqlite3

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "context").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from this project or notebooks/.")


repo_root = find_repo_root(Path.cwd().resolve())
sqlite_path = repo_root / "data" / "uni_rag.sqlite"
if not sqlite_path.is_file():
    raise FileNotFoundError(
        f"Extraction database not found at {sqlite_path}. Run "
        "`uv run -m uni_rag_agent storage init`, then "
        "`uv run -m uni_rag_agent inventory run`, then "
        "`uv run -m uni_rag_agent extract run`."
    )
sqlite_uri = sqlite_path.resolve().as_uri() + "?mode=ro"

connection = sqlite3.connect(sqlite_uri, uri=True, timeout=5)
connection.execute("PRAGMA query_only=ON")

In [5]:
latest_runs = pd.read_sql_query(
    """
    SELECT id, started_at, finished_at, status, config_json,
           files_seen, files_indexed, files_failed, error
    FROM extraction_runs
    ORDER BY id DESC
    LIMIT 20
    """,
    connection,
)
latest_runs

,id,started_at,finished_at,status,config_json,files_seen,files_indexed,files_failed,error
0,2,2026-06-27T16:51:54.551331+00:00,2026-06-27T16:58:46.656367+00:00,completed,"{""category"": null, ""courses_root"": ""D:\\Projec...",1482,1337,145,None
1,1,2026-06-25T21:22:38.461169+00:00,2026-06-25T21:42:58.532961+00:00,completed,"{""classification_categories"": [""archive_metada...",33912,0,0,None


In [6]:
extraction_docs = pd.read_sql_query(
    """
    SELECT extracted_documents.id AS extracted_document_id,
           files.id AS file_id,
           courses.name AS course_name,
           files.relative_path,
           files.extension,
           files.category,
           files.index_status,
           extracted_documents.extractor_name,
           extracted_documents.status AS extraction_status,
           extracted_documents.text_length,
           extracted_documents.chunk_count,
           extracted_documents.error,
           extracted_documents.extracted_at
    FROM extracted_documents
    JOIN files ON files.id = extracted_documents.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    ORDER BY extracted_documents.extracted_at DESC, files.relative_path
    """,
    connection,
)
extraction_docs.head(25)

,extracted_document_id,file_id,course_name,relative_path,extension,category,index_status,extractor_name,extraction_status,text_length,chunk_count,error,extracted_at
0,1482,33912,Uni Textbooks,Uni Textbooks\Starting_Out_with_Programming_Lo...,.pdf,document,indexed,pdf-pymupdf,indexed,1302039,971,NaN,2026-06-27T16:58:46.526673+00:00
1,1481,33911,Uni Textbooks,Uni Textbooks\rosen_discrete_mathematics_and_i...,.pdf,document,indexed,pdf-pymupdf,indexed,3569454,1166,NaN,2026-06-27T16:58:39.106476+00:00
2,1480,33910,Uni Textbooks,Uni Textbooks\Passages Level 2 Students Book (...,.pdf,document,indexed,pdf-pymupdf,indexed,367306,152,NaN,2026-06-27T16:58:25.484288+00:00
3,1479,33909,Uni Textbooks,Uni Textbooks\Foundations of CS 4th Edition- B...,.pdf,document,indexed,pdf-pymupdf,indexed,1254629,681,NaN,2026-06-27T16:58:10.277168+00:00
4,1478,33908,Uni Textbooks,Uni Textbooks\037-Elementary-Linear-Algebra-Ap...,.pdf,document,indexed,pdf-pymupdf,indexed,1835209,793,NaN,2026-06-27T16:57:57.586382+00:00
5,1477,33907,Techincal Writing,Techincal Writing\Worksheets\worksheet4.docx,.docx,document,indexed,docx-python-docx,indexed,1360,1,NaN,2026-06-27T16:57:51.755333+00:00
6,1476,33906,Techincal Writing,Techincal Writing\Worksheets\Worksheet2 soluti...,.docx,document,indexed,docx-python-docx,indexed,2507,1,NaN,2026-06-27T16:57:51.682099+00:00
7,1475,33905,Techincal Writing,Techincal Writing\Worksheets\Worksheet 6.docx,.docx,document,indexed,docx-python-docx,indexed,1304,1,NaN,2026-06-27T16:57:51.616725+00:00
8,1474,33904,Techincal Writing,Techincal Writing\Worksheets\Worksheet 5.docx,.docx,document,indexed,docx-python-docx,indexed,1263,1,NaN,2026-06-27T16:57:51.567864+00:00
9,1473,33903,Techincal Writing,Techincal Writing\Worksheets\Worksheet 3.docx,.docx,document,indexed,docx-python-docx,indexed,3665,1,NaN,2026-06-27T16:57:51.519500+00:00


In [7]:
coverage_by_category = pd.read_sql_query(
    """
    SELECT files.category,
           files.extension,
           files.index_status,
           COUNT(*) AS file_count
    FROM files
    WHERE files.category IN ('document', 'slides', 'notebook', 'code', 'transcript')
    GROUP BY files.category, files.extension, files.index_status
    ORDER BY files.category, files.extension, files.index_status
    """,
    connection,
)
coverage_by_category

,category,extension,index_status,file_count
0,code,.cpp,indexed,21
1,code,.h,indexed,4
2,code,.m,indexed,5
3,code,.py,failed,13
4,code,.py,indexed,120
5,code,.r,indexed,45
6,document,.doc,failed,15
7,document,.docx,failed,1
8,document,.docx,indexed,112
9,document,.md,indexed,10


In [8]:
chunks = pd.read_sql_query(
    """
    SELECT chunks.id AS chunk_id,
           files.id AS file_id,
           courses.name AS course_name,
           files.relative_path,
           files.extension,
           chunks.source_type,
           chunks.location_type,
           chunks.location_value,
           chunks.token_count,
           LENGTH(chunks.text) AS text_chars
    FROM chunks
    JOIN files ON files.id = chunks.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    ORDER BY files.relative_path, chunks.chunk_index
    """,
    connection,
)
chunks.head(25)

,chunk_id,file_id,course_name,relative_path,extension,source_type,location_type,location_value,token_count,text_chars
0,1,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,1,13,95
1,2,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,2,4,27
2,3,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,3,40,254
3,4,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,4,43,271
4,5,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,5,65,396
5,6,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,6,36,220
6,7,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,7,62,390
7,8,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,8,68,366
8,9,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,9,59,279
9,10,4,Algorithms Design and Analysis,Algorithms Design and Analysis\1-Graphs\2 grap...,.pdf,document,page,10,84,409


In [9]:
chunk_source_summary = (
    chunks.groupby(["source_type", "location_type"], dropna=False)
    .agg(chunk_count=("chunk_id", "count"), median_tokens=("token_count", "median"), max_tokens=("token_count", "max"))
    .reset_index()
    .sort_values(["source_type", "location_type"])
)
chunk_source_summary

,source_type,location_type,chunk_count,median_tokens,max_tokens
0,code,class,137,45.0,839
1,code,function,594,38.0,991
2,code,module,287,26.0,823
3,code,subchunk,22,1000.0,1000
4,document,docx_section,137,413.0,1000
5,document,markdown_section,103,46.0,279
6,document,page,17436,123.0,998
7,document,subchunk,224,986.5,1000
8,document,text_section,397,1.0,1000
9,notebook,notebook_cell,3802,23.0,673


In [10]:
failed_extractions = extraction_docs.loc[extraction_docs["extraction_status"] == "failed"]
failed_extractions[["course_name", "relative_path", "extension", "extractor_name", "error", "extracted_at"]].head(50)

,course_name,relative_path,extension,extractor_name,error,extracted_at
25,Structured Programming,Structured Programming\Worksheet-Recursion.doc,.doc,legacy-doc-unsupported,legacy format not supported yet,2026-06-27T16:57:46.729654+00:00
88,Stats for Data Science,Stats for Data Science\Counting Methods and Pr...,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:57:31.216143+00:00
100,Stat Methods,Stat Methods\Testing HypothesesFile.pdf,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:10.640120+00:00
102,Stat Methods,Stat Methods\Normal distribution part 2File.pdf,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:10.376440+00:00
103,Stat Methods,Stat Methods\Lecture 19( Normal Distribution P...,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:10.299478+00:00
104,Stat Methods,"Stat Methods\Lecture 18 ( Binomial, Geometric,...",.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:10.207202+00:00
105,Stat Methods,Stat Methods\Lecture 17 ( continuous random va...,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:10.085360+00:00
106,Stat Methods,Stat Methods\Lecture 16( Discrete Random Varia...,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:09.975167+00:00
107,Stat Methods,Stat Methods\Lecture 15 (Discrete Random Varia...,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:09.884985+00:00
108,Stat Methods,Stat Methods\Lecture 14 ( Counting Rules Part ...,.pdf,pdf-pymupdf,"scanned PDF, OCR not available",2026-06-27T16:56:09.811496+00:00
